In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import duckdb
import pandas as pd

# Project paths
DATA_DIR = Path("../data")
REPORTS_DIR = Path("../reports")

# Load the cleaned dataset
DATA_PATH = DATA_DIR / "subway_ml_dataset.parquet"

print("DuckDB version:", duckdb.__version__)
print("Dataset exists:", DATA_PATH.exists())

# Connect to a local DuckDB database
con = duckdb.connect(str(REPORTS_DIR / "subway_analytics.duckdb"))

# Register the Parquet dataset as a SQL view
con.execute(f"""
    CREATE OR REPLACE VIEW subway_segments AS
    SELECT *
    FROM read_parquet('{DATA_PATH.as_posix()}')
""")

# Check the data
display(con.execute("SELECT * FROM subway_segments LIMIT 5").df())

DuckDB version: 2.0.0
Dataset exists: True


,service_date,a_station,b_station,route_ids,day_of_week,month,day_of_month,mean_seconds
0,2025-01-01,Van Cortlandt Park-242 St,238 St,1,2,1,1,97.30
1,2025-01-01,238 St,Van Cortlandt Park-242 St,1,2,1,1,174.16
2,2025-01-01,238 St,231 St,1,2,1,1,87.01
3,2025-01-01,231 St,Van Cortlandt Park-242 St,1,2,1,1,214.93
4,2025-01-01,231 St,238 St,1,2,1,1,128.88


In [4]:
monthly_analysis = con.execute("""
    SELECT
        EXTRACT(MONTH FROM service_date) AS month_number,
        STRFTIME(service_date, '%B') AS month_name,
        COUNT(*) AS segment_records,
        COUNT(DISTINCT service_date) AS service_days,
        ROUND(AVG(mean_seconds), 2) AS avg_travel_seconds,
        ROUND(AVG(mean_seconds) / 60.0, 2) AS avg_travel_minutes,
        ROUND(MEDIAN(mean_seconds) / 60.0, 2) AS median_travel_minutes
    FROM subway_segments
    GROUP BY month_number, month_name
    ORDER BY month_number
""").df()

display(monthly_analysis)

,month_number,month_name,segment_records,service_days,avg_travel_seconds,avg_travel_minutes,median_travel_minutes
0,1,January,26112,23,139.37,2.32,1.83
1,2,February,22551,20,138.42,2.31,1.82
2,3,March,23390,21,136.68,2.28,1.82
3,4,April,25045,22,139.67,2.33,1.84
4,5,May,24957,22,139.32,2.32,1.84
5,6,June,24055,21,141.29,2.35,1.84
6,7,July,26128,23,140.31,2.34,1.84
7,8,August,24032,21,139.38,2.32,1.82
8,9,September,25014,22,138.63,2.31,1.82
9,10,October,26237,23,139.37,2.32,1.83


In [5]:
slowest_segments = con.execute("""
    SELECT
        a_station,
        b_station,
        route_ids,
        COUNT(*) AS record_count,
        ROUND(AVG(mean_seconds) / 60.0, 2) AS avg_travel_minutes,
        ROUND(MEDIAN(mean_seconds) / 60.0, 2) AS median_travel_minutes
    FROM subway_segments
    GROUP BY a_station, b_station, route_ids
    HAVING COUNT(*) >= 100
    ORDER BY avg_travel_minutes DESC
    LIMIT 15
""").df()

display(slowest_segments)

,a_station,b_station,route_ids,record_count,avg_travel_minutes,median_travel_minutes
0,St George,Great Kills,SS,253,16.87,16.84
1,3 Av-149 St,E 180 St,5,197,11.78,11.61
2,E 180 St,3 Av-149 St,5,239,11.43,11.03
3,Canal St,Atlantic Av-Barclays Ctr,N,256,11.04,11.02
4,75 St-Elderts Ln,Sutphin Blvd-Archer Av-JFK Airport,J,172,10.49,10.49
5,Woodhaven Blvd,Van Siclen Av,J Z,100,10.46,10.47
6,Atlantic Av-Barclays Ctr,Canal St,N,251,10.31,10.25
7,Woodhaven Blvd,Crescent St,J,187,10.25,10.24
8,Woodhaven Blvd,Norwood Av,J,190,10.24,10.22
9,Woodhaven Blvd,Cleveland St,J,254,10.22,10.23


In [6]:
variable_segments = con.execute("""
    SELECT
        a_station,
        b_station,
        route_ids,
        COUNT(*) AS record_count,
        ROUND(AVG(mean_seconds) / 60.0, 2) AS avg_travel_minutes,
        ROUND(STDDEV_SAMP(mean_seconds), 2) AS std_dev_seconds,
        ROUND(
            100.0 * STDDEV_SAMP(mean_seconds)
            / NULLIF(AVG(mean_seconds), 0),
            2
        ) AS variability_pct
    FROM subway_segments
    GROUP BY a_station, b_station, route_ids
    HAVING COUNT(*) >= 100
    ORDER BY variability_pct DESC
    LIMIT 15
""").df()

display(variable_segments)

,a_station,b_station,route_ids,record_count,avg_travel_minutes,std_dev_seconds,variability_pct
0,Botanic Garden,Prospect Park,FS,110,2.12,74.27,58.40
1,Shepherd Av,Euclid Av,A C,259,0.99,26.64,44.99
2,Avenue U,Kings Hwy,F,107,2.11,49.51,39.12
3,Crescent St,Cypress Hills,J Z,113,3.93,86.07,36.50
4,Parsons Blvd,169 St,F,105,3.10,66.57,35.82
5,20 Av,Bay Pkwy,D,168,1.42,29.66,34.74
6,Aqueduct-N Conduit Av,Howard Beach-JFK Airport,A,257,1.84,36.14,32.72
7,86 St,Bay Ridge-95 St,R,244,0.62,11.63,31.17
8,Junction Blvd,103 St-Corona Plaza,7,235,1.03,18.79,30.53
9,Woodhaven Blvd,63 Dr-Rego Park,E F M R,168,2.10,37.94,30.12


In [7]:
variable_segments = con.execute("""
    SELECT
        a_station,
        b_station,
        route_ids,
        COUNT(*) AS record_count,
        ROUND(AVG(mean_seconds) / 60.0, 2) AS avg_travel_minutes,
        ROUND(STDDEV_SAMP(mean_seconds), 2) AS std_dev_seconds,
        ROUND(
            100.0 * STDDEV_SAMP(mean_seconds)
            / NULLIF(AVG(mean_seconds), 0),
            2
        ) AS variability_pct
    FROM subway_segments
    GROUP BY a_station, b_station, route_ids
    HAVING COUNT(*) >= 100
    ORDER BY variability_pct DESC
    LIMIT 15
""").df()

display(variable_segments)

,a_station,b_station,route_ids,record_count,avg_travel_minutes,std_dev_seconds,variability_pct
0,Botanic Garden,Prospect Park,FS,110,2.12,74.27,58.40
1,Shepherd Av,Euclid Av,A C,259,0.99,26.64,44.99
2,Avenue U,Kings Hwy,F,107,2.11,49.51,39.12
3,Crescent St,Cypress Hills,J Z,113,3.93,86.07,36.50
4,Parsons Blvd,169 St,F,105,3.10,66.57,35.82
5,20 Av,Bay Pkwy,D,168,1.42,29.66,34.74
6,Aqueduct-N Conduit Av,Howard Beach-JFK Airport,A,257,1.84,36.14,32.72
7,86 St,Bay Ridge-95 St,R,244,0.62,11.63,31.17
8,Junction Blvd,103 St-Corona Plaza,7,235,1.03,18.79,30.53
9,Woodhaven Blvd,63 Dr-Rego Park,E F M R,168,2.10,37.94,30.12


In [8]:
weekday_analysis = con.execute("""
    SELECT
        CASE
            WHEN day_of_week = 0 THEN 'Monday'
            WHEN day_of_week = 1 THEN 'Tuesday'
            WHEN day_of_week = 2 THEN 'Wednesday'
            WHEN day_of_week = 3 THEN 'Thursday'
            WHEN day_of_week = 4 THEN 'Friday'
            ELSE 'Other'
        END AS weekday,
        COUNT(*) AS segment_records,
        ROUND(AVG(mean_seconds) / 60.0, 2) AS avg_travel_minutes,
        ROUND(MEDIAN(mean_seconds) / 60.0, 2) AS median_travel_minutes
    FROM subway_segments
    GROUP BY day_of_week
    ORDER BY day_of_week
""").df()

display(weekday_analysis)

,weekday,segment_records,avg_travel_minutes,median_travel_minutes
0,Monday,58767,2.30,1.82
1,Tuesday,59196,2.33,1.84
2,Wednesday,60242,2.33,1.84
3,Thursday,58963,2.32,1.83
4,Friday,58946,2.31,1.83


In [9]:
segment_coverage = con.execute("""
    SELECT
        a_station,
        b_station,
        route_ids,
        COUNT(*) AS record_count,
        COUNT(DISTINCT service_date) AS observed_days,
        ROUND(AVG(mean_seconds) / 60.0, 2) AS avg_travel_minutes
    FROM subway_segments
    GROUP BY a_station, b_station, route_ids
    ORDER BY record_count DESC
    LIMIT 15
""").df()

display(segment_coverage)

,a_station,b_station,route_ids,record_count,observed_days,avg_travel_minutes
0,Mt Eden Av,176 St,4,261,261,1.30
1,Prospect Av,Jackson Av,2 5,261,261,1.65
2,174 St,Freeman St,2 5,261,261,1.71
3,DeKalb Av,Myrtle-Wyckoff Avs,L,261,261,1.74
4,Wilson Av,Halsey St,L,261,261,1.99
5,Sutter Av,Livonia Av,L,261,261,1.55
6,Bedford Park Blvd-Lehman College,Mosholu Pkwy,4,261,261,1.94
7,Rockaway Blvd,88 St,A,261,261,1.46
8,183 St,Fordham Rd,4,261,261,1.42
9,Graham Av,Lorimer St,L,261,261,1.50


In [10]:
from pathlib import Path

REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sql_results = {
    "monthly_analysis": monthly_analysis,
    "slowest_segments": slowest_segments,
    "variable_segments": variable_segments,
    "weekday_analysis": weekday_analysis,
    "segment_coverage": segment_coverage
}

for name, result_df in sql_results.items():
    output_path = REPORTS_DIR / f"{name}.csv"
    result_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path.resolve()}")

print("\nAll SQL analysis results exported successfully!")

Saved: C:\Users\Pradeep\OneDrive\Desktop\projects\NYC-Subway-Analytics\reports\monthly_analysis.csv
Saved: C:\Users\Pradeep\OneDrive\Desktop\projects\NYC-Subway-Analytics\reports\slowest_segments.csv
Saved: C:\Users\Pradeep\OneDrive\Desktop\projects\NYC-Subway-Analytics\reports\variable_segments.csv
Saved: C:\Users\Pradeep\OneDrive\Desktop\projects\NYC-Subway-Analytics\reports\weekday_analysis.csv
Saved: C:\Users\Pradeep\OneDrive\Desktop\projects\NYC-Subway-Analytics\reports\segment_coverage.csv

All SQL analysis results exported successfully!
